Обработка записанных логов при движения КС по угольному разрезу

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib; matplotlib.use("QtAgg")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

PATH_DATA = PROJECT_ROOT / "data"
DATA_RECORD = "2024.09.05_utro"

PATH_REC = PATH_DATA / "Records" / DATA_RECORD
PATH_OUT = PATH_DATA / "Processed" / DATA_RECORD

PATH_GNSS0 = PATH_REC / "gnss0_data.csv"
PATH_GNSS1 = PATH_REC / "gnss1_data.csv"
PATH_IMU = PATH_REC / "imu_data.csv"
PATH_VEL = PATH_REC / "vel_data.csv"

In [ ]:
print(pd.read_csv(PATH_IMU, nrows=5).columns.tolist())
print(pd.read_csv(PATH_GNSS0, nrows=5).columns.tolist())
print(pd.read_csv(PATH_VEL, nrows=5).columns.tolist())

Выгружаем лог в parquet + различные преобразования (сетка, t0, поворот на константное смещение и т.д.)

In [ ]:
import src.preprocessing.preprocess as pp

out = pp.run(
    rec_dir = PATH_REC,
    out_dir = PATH_OUT,
    imu_tilt_deg = (16.5, 0.0, -0.5),     # (Ox, Oy, Oz) в градусах
    warmup_sec = 30,
)
gnss0, imu = out["gnss0"], out["imu"]

In [ ]:
P = Path("/home/rsadovec/prj/KAMAZ_Jupyter/data/Processed/2024.09.05_utro")

for nm in ["gnss0", "vel0", "imu"]:
    df = pd.read_parquet(P / f"{nm}.parquet")
    print(f"\n=== {nm} ===")
    print("shape:", df.shape)
    print(df.head(3).to_string())
    print("t range:", df['t'].min(), "->", df['t'].max())

In [2]:
%matplotlib qt

plt.show()

ImportError: Cannot load backend 'qtagg' which requires the 'qt' interactive framework, as 'headless' is currently running

Нарезка сценариев по логам

In [ ]:
%matplotlib qt

import src.preprocessing.segmentation as sg

PROC = Path("/home/rsadovec/prj/KAMAZ_Jupyter/data/Processed/2024.09.05_utro")
data = sg.load_processed(PROC)

sg.overview(data)              # вся запись: скорость, RMS вибраций, RMS гироскопа, высота
sg.zoom(data, 6500, 8200)      # зум в подозрительный интервал, сырой IMU для точных границ

segments = [
    sg.Seg(0,    580,  "stop_EngOff"),
    sg.Seg(600,  950,  "stop_EngOn"),
    sg.Seg(950,  3030, "dvij"),
    sg.Seg(6600, 8100, "stop_EngOff"),
    sg.Seg(10320,10570,"gruz"),
]

sg.validate_segments(segments, total_dur=data["imu"]["t"].max())  # ловит пересечения
sg.overview(data, segments=segments)   # подсветка размеченного цветом — визуальный контроль
sg.cut_and_save(data, segments, PROC / "segments")